# 01 – Exploratory Data Analysis

This notebook explores the OPSSAT-AD dataset: distributions, correlations, missing values and anomaly prevalence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.utils.preprocessing import generate_synthetic_dataset, FEATURE_COLUMNS

# Generate (or load) the dataset
df = generate_synthetic_dataset(n_samples=120_000, anomaly_ratio=0.038, seed=42)
print(df.shape)
df.head()

In [ ]:
# ── Label distribution ──────────────────────────────────────────────────────
print('Label counts:')
print(df['label'].value_counts())
print(f'Anomaly ratio: {df["label"].mean():.4f}')

In [ ]:
# ── Feature distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 5, figsize=(20, 12))
for ax, col in zip(axes.flat, FEATURE_COLUMNS):
    df[col].hist(ax=ax, bins=50, color='steelblue', edgecolor='none')
    ax.set_title(col, fontsize=8)
plt.tight_layout()
plt.savefig('../results/eda_feature_distributions.png', dpi=100)
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr = df[FEATURE_COLUMNS].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, square=True,
            linewidths=0.5, annot=False)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../results/eda_correlation_heatmap.png', dpi=100)
plt.show()

In [ ]:
# ── Time series sample plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)
for ax, col in zip(axes, ['temperature_1', 'voltage_1', 'snr_1']):
    ax.plot(df['timestamp'][:5000], df[col][:5000], lw=0.8, label=col)
    anomaly_ts = df.loc[df['label'] == 1, 'timestamp'][:5000]
    anomaly_vals = df.loc[df['label'] == 1, col][:5000]
    ax.scatter(anomaly_ts, anomaly_vals, color='red', s=10, zorder=5,
               label='anomaly')
    ax.set_ylabel(col)
    ax.legend(fontsize=7)
axes[-1].set_xlabel('Time')
plt.suptitle('Telemetry Signals (first 5,000 points)')
plt.tight_layout()
plt.savefig('../results/eda_time_series_sample.png', dpi=100)
plt.show()